# Siding Repair: Subtopic Updater

For a given cluster stub, generates content for every `*Content to be generated.*` subtopic section
and writes the results back to the file — including a consolidated `## References` table.

Each generation call receives:
- The **full document** as context (so the writer sees the page scope and existing sections)
- **Top-K RAG passages** from the construction books LanceDB

Superscript references are renumbered globally across all subtopics before writing.

In [1]:
from pathlib import Path

LANCEDB_PATH = r"C:\Users\tfalcon\microsites\tools\programatic writing stuffs\DBA writer\lancedb_construction_books"
LANCEDB_TABLE = "construction_books"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MODEL = "gpt-4o"
TEMPERATURE = 0.7
TOP_K = 20
WORD_TARGET = 200

# Target file — change these to process a different cluster
TARGET_SLUG = "hardie-fiber-cement-siding-repair"
TARGET_LOCATION = "portland"

SIDING_REPAIR_DIR = Path("../../../apps/siding-repair/src/data/generated_content")
AGENTS_DIR = Path("../agents")
CONTENT_REVIEW_DIR = Path("../../content-review")

In [2]:
import sys, os, re
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(CONTENT_REVIEW_DIR.resolve()))

import lancedb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from agent_loader import load_agent

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embedder = SentenceTransformer(EMBEDDING_MODEL)
db = lancedb.connect(LANCEDB_PATH)
table = db.open_table(LANCEDB_TABLE)
agent = load_agent(AGENTS_DIR / "01-subtopic-writer.md")
print(f"LanceDB: {table.count_rows()} docs | Agent: {agent.name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LanceDB: 6260 docs | Agent: Subtopic Writer


In [3]:
# Find and load the target stub file
filename = f"service_page_cluster_{TARGET_SLUG}_{TARGET_LOCATION}.md"
stub_path = SIDING_REPAIR_DIR / filename

if not stub_path.exists():
    raise FileNotFoundError(f"Stub not found: {stub_path}")

doc_text = stub_path.read_text(encoding="utf-8")

# Parse subtopics from CLUSTER_META
meta_match = re.search(r'<!--\s*CLUSTER_META([\s\S]*?)-->', doc_text)
if not meta_match:
    raise ValueError("No CLUSTER_META block found")

subtopics = []
in_subtopics = False
for line in meta_match.group(1).split('\n'):
    s = line.strip()
    if s.startswith('subtopics:'):
        in_subtopics = True; continue
    if in_subtopics:
        if s.startswith('- '): subtopics.append(s[2:].strip())
        elif s and not s.startswith('-'): in_subtopics = False

# Identify which subtopics still need content (normalize line endings)
doc_normalized = doc_text.replace('\r\n', '\n')
pending = [t for t in subtopics if f"## {t}\n*Content to be generated.*" in doc_normalized]

print(f"File: {filename}")
print(f"Subtopics total: {len(subtopics)} | Pending: {len(pending)}")
for t in subtopics:
    status = "PENDING" if t in pending else "done"
    print(f"  [{status}] {t}")

File: service_page_cluster_hardie-fiber-cement-siding-repair_portland.md
Subtopics total: 1 | Pending: 1
  [PENDING] Hardie Siding Repair


In [4]:
def generate_subtopic(subtopic: str, doc_text: str, service: str, location_full: str) -> tuple:
    """
    Returns (content_body, ref_rows) where:
      content_body  — markdown section body with <sup>1</sup> / <sup>2</sup>
      ref_rows      — list of pipe-delimited ref rows: ['| 1 | ... |', '| 2 | ... |']
    """
    # RAG retrieval
    query = f"{subtopic} {service} {location_full}"
    results = table.search(embedder.encode([query])[0]).limit(TOP_K).to_pandas()
    rag_context = "\n\n---\n\n".join(
        f"Source: {row.get('source','?')} (Page {row.get('page','N/A')})\n{row['text']}"
        for _, row in results.iterrows()
    )

    user_prompt = f"""## Full document context (understand the page scope before writing):

{doc_text}

---

## Construction reference material (RAG):

{rag_context}

---

Write a ~{WORD_TARGET}-word section body for:

Subtopic: {subtopic}
Parent page: {service} — {location_full}

Rules:
- Do NOT include the section heading
- Use <sup>1</sup> and <sup>2</sup> for inline citations
- After the section body, output exactly one line: <!-- REFS -->
- Then output exactly 2 pipe-delimited reference rows (no header row), format:
  | N | Source Title | p. X | Brief note |
- Return nothing else"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": agent.system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    raw = response.choices[0].message.content.strip()

    # Split on <!-- REFS -->
    parts = raw.split("<!-- REFS -->", 1)
    content_body = parts[0].strip()
    ref_rows = []
    if len(parts) == 2:
        ref_rows = [l.strip() for l in parts[1].strip().splitlines() if l.strip().startswith("|")]

    return content_body, ref_rows

In [5]:
location_full = "Portland, Oregon" if TARGET_LOCATION == "portland" else "Seattle, Washington"

# Re-read fresh so we pass the current file state
doc_text = stub_path.read_text(encoding="utf-8")

# Service label from meta block
service_match = re.search(r'service:\s*(.+)', doc_text)
service_label = service_match.group(1).strip() if service_match else "siding repair"

results_store = {}  # subtopic -> (content_body, ref_rows)

for i, subtopic in enumerate(pending):
    print(f"[{i+1}/{len(pending)}] Generating: {subtopic} ...", end=" ", flush=True)
    content, refs = generate_subtopic(subtopic, doc_text, service_label, location_full)
    results_store[subtopic] = (content, refs)
    words = len(content.split())
    print(f"{words} words, {len(refs)} refs")

print("\nDone.")

[1/1] Generating: Hardie Siding Repair ... 167 words, 2 refs

Done.


In [6]:
# Preview all generated content before writing
for subtopic, (content, refs) in results_store.items():
    print(f"\n{'='*60}")
    print(f"## {subtopic}")
    print(f"{'='*60}")
    print(content)
    if refs:
        print("\n--- refs ---")
        for r in refs:
            print(r)


## Hardie Siding Repair
Hardie siding, a popular fiber cement option, is known for its durability and resistance to the elements, making it a preferred choice for Portland homeowners. However, like any siding, it can still suffer damage from impact, poor installation, or age. Repairing Hardie siding involves a few essential steps to ensure longevity and aesthetic appeal. First, it's crucial to identify and remove any damaged sections carefully to avoid harming the weather-resistive barrier beneath<sup>1</sup>. When replacing or patching, use corrosion-resistant nails and ensure all joints are adequately sealed to prevent moisture infiltration<sup>2</sup>. Properly sealing the end grains and face nails with paint or sealant is also vital to protect against moisture and maintain the siding's finish. Given the brittle nature of fiber cement, using appropriate tools like power shears or carbide-tipped scoring tools can prevent cracking during repairs<sup>2</sup>. If you're unsure about th

In [7]:
def renumber_refs(text: str, offset: int) -> str:
    """Shift <sup>1</sup> and <sup>2</sup> by offset."""
    text = re.sub(r'<sup>1</sup>', f'<sup>{offset + 1}</sup>', text)
    text = re.sub(r'<sup>2</sup>', f'<sup>{offset + 2}</sup>', text)
    return text

def renumber_ref_rows(rows: list, offset: int) -> list:
    """Renumber the leading | N | in each ref row."""
    out = []
    for row in rows:
        row = re.sub(r'^\|\s*\d+\s*\|', f'| {offset + len(out) + 1} |', row)
        out.append(row)
    return out

updated = stub_path.read_text(encoding="utf-8").replace('\r\n', '\n')
all_ref_rows = []
global_offset = 0

for subtopic, (content, refs) in results_store.items():
    numbered_content = renumber_refs(content, global_offset)
    numbered_refs = renumber_ref_rows(refs, global_offset)
    all_ref_rows.extend(numbered_refs)
    global_offset += len(refs)

    placeholder = f"## {subtopic}\n*Content to be generated.*"
    replacement = f"## {subtopic}\n{numbered_content}"
    updated = updated.replace(placeholder, replacement)

# Build references table
if all_ref_rows:
    ref_table = (
        "\n## References\n\n"
        "| # | Source | Page | Note |\n"
        "|---|--------|------|------|\n"
        + "\n".join(all_ref_rows)
        + "\n"
    )
    if "## Page Metadata" in updated:
        updated = updated.replace("## Page Metadata", ref_table + "\n## Page Metadata")
    else:
        updated += ref_table

# Mark status as draft
updated = re.sub(r'(status:\s*)stub', r'\1draft', updated)

stub_path.write_text(updated, encoding="utf-8")
print(f"Written: {stub_path.name}")
print(f"Sections filled: {len(results_store)} | References added: {len(all_ref_rows)}")

Written: service_page_cluster_hardie-fiber-cement-siding-repair_portland.md
Sections filled: 1 | References added: 2


In [8]:
# Verify — print the updated file
print(stub_path.read_text(encoding="utf-8"))

# Hardie & Fiber Cement Siding Repair - Portland, Oregon

<!-- CLUSTER_META
service: siding-repair
cluster_id: 3
cluster_slug: hardie-fiber-cement-siding-repair
location: portland
status: draft
subtopics:
  - Hardie Siding Repair
-->

## Hero Section

### [STUB] Hardie & Fiber Cement Siding Repair in Portland, Oregon
*Content to be generated.*

## Hardie Siding Repair
Hardie siding, a popular fiber cement option, is known for its durability and resistance to the elements, making it a preferred choice for Portland homeowners. However, like any siding, it can still suffer damage from impact, poor installation, or age. Repairing Hardie siding involves a few essential steps to ensure longevity and aesthetic appeal. First, it's crucial to identify and remove any damaged sections carefully to avoid harming the weather-resistive barrier beneath<sup>1</sup>. When replacing or patching, use corrosion-resistant nails and ensure all joints are adequately sealed to prevent moisture infiltration<su